### ------------------Clean World Bank Data Notebook-----------------------
This notebook cleans the raw World Bank country-year dataset** and creates a processed, analytics-ready World Bank file.

In [2]:
# Import necessary libraries
import os
from datetime import datetime
import numpy as np
import pandas as pd

In [3]:
# Creating variable for file paths and output directories.

RAW_WORLD_BANK_FILE = r"C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\raw_data\world_bank_data\world_bank_country_year_2015_2024_20260707_222046.csv"
PROCESSED_OUTPUT_DIR = r"C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\processed_data"

os.makedirs(PROCESSED_OUTPUT_DIR, exist_ok=True)

print("Raw World Bank file path:")
print(RAW_WORLD_BANK_FILE)

print("\nProcessed output folder:")
print(PROCESSED_OUTPUT_DIR)

Raw World Bank file path:
C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\raw_data\world_bank_data\world_bank_country_year_2015_2024_20260707_222046.csv

Processed output folder:
C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\processed_data


In [4]:
# Loading the raw World Bank dataset
df_raw = pd.read_csv(RAW_WORLD_BANK_FILE)

print("Raw World Bank dataset loaded successfully.")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")

Raw World Bank dataset loaded successfully.
Rows: 2,170
Columns: 7


In [5]:
df_raw.head()

,reporter_iso,reporter_name,year,gdp_current_usd,population_total,gdp_growth_annual_pct,trade_pct_of_gdp
0,ABW,Aruba,2015,2.962907e+09,107906,3.611479,140.493001
1,ABW,Aruba,2016,2.983637e+09,108727,1.234335,136.887263
2,ABW,Aruba,2017,3.092428e+09,108735,3.493430,138.716620
3,ABW,Aruba,2018,3.276188e+09,108908,3.212471,138.824972
4,ABW,Aruba,2019,3.347562e+09,109203,1.229144,137.923249


#### Understanding the raw data structure

In [6]:
# Displaying the column names of the raw dataset
df_raw.columns.tolist()

['reporter_iso',
 'reporter_name',
 'year',
 'gdp_current_usd',
 'population_total',
 'gdp_growth_annual_pct',
 'trade_pct_of_gdp']

In [7]:
# Checking info of the raw dataset
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2170 entries, 0 to 2169
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   reporter_iso           2170 non-null   str    
 1   reporter_name          2170 non-null   str    
 2   year                   2170 non-null   int64  
 3   gdp_current_usd        2088 non-null   float64
 4   population_total       2170 non-null   int64  
 5   gdp_growth_annual_pct  2084 non-null   float64
 6   trade_pct_of_gdp       1820 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 118.8 KB


In [8]:
# Getting summary statistics of the raw dataset
df_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
reporter_iso,2170,217,ABW,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reporter_name,2170,217,Aruba,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,2170.0,NaN,NaN,NaN,2019.5,2.872943,2015.0,2017.0,2019.5,2022.0,2024.0
gdp_current_usd,2088.0,NaN,NaN,NaN,436064674618.632812,1994366436277.252441,36811936.08,6896774250.0,29373362838.5,193187750000.0,29298000000000.0
population_total,2170.0,NaN,NaN,NaN,35846768.660369,139073594.379218,9646.0,782395.75,6455846.5,25329112.75,1450935791.0
gdp_growth_annual_pct,2084.0,NaN,NaN,NaN,2.630794,6.427663,-54.402093,0.926962,3.102713,5.047026,75.308418
trade_pct_of_gdp,1820.0,NaN,NaN,NaN,91.161039,58.017125,1.995412,53.599053,78.020784,110.23825,412.17717


In [9]:
# Checking for duplicate rows 
duplicate_rows = df_raw.duplicated().sum()
print(f"Exact duplicate rows in raw dataset: {duplicate_rows:,}")

Exact duplicate rows in raw dataset: 0


In [10]:
# Checking for missing values in the raw dataset
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_summary

,missing_count,missing_pct
trade_pct_of_gdp,350,16.13
gdp_growth_annual_pct,86,3.96
gdp_current_usd,82,3.78
year,0,0.00
reporter_name,0,0.00
reporter_iso,0,0.00
population_total,0,0.00


#### Checking the country-year structure

In [11]:
# Checking for unique values in key columns
print("Unique country codes:", df_raw["reporter_iso"].nunique(dropna=True))
print("Unique country names:", df_raw["reporter_name"].nunique(dropna=True))
print("Unique years:", sorted(df_raw["year"].dropna().unique().tolist()))

Unique country codes: 217
Unique country names: 217
Unique years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [12]:
# Checking for duplicate country-year combinations
country_year_duplicates = df_raw.duplicated(subset=["reporter_iso", "year"]).sum()
print(f"Duplicate country-year rows: {country_year_duplicates:,}")

Duplicate country-year rows: 0


In [13]:
# Standardizing column names for consistency and clarity
column_rename_map = {
    "reporter_iso": "reporter_iso3",
    "reporter_name": "reporter_name",
    "year": "reference_year",
    "gdp_current_usd": "gdp_current_usd",
    "population_total": "population_total",
    "gdp_growth_annual_pct": "gdp_growth_annual_pct",
    "trade_pct_of_gdp": "trade_pct_of_gdp"
}

df = df_raw.rename(columns=column_rename_map).copy()

print("Column names standardized successfully.")
print(df.columns.tolist())

Column names standardized successfully.
['reporter_iso3', 'reporter_name', 'reference_year', 'gdp_current_usd', 'population_total', 'gdp_growth_annual_pct', 'trade_pct_of_gdp']


#### Keeping the columns needed for this project

In [14]:
columns_to_keep = [
    "reporter_iso3",
    "reporter_name",
    "reference_year",
    "gdp_current_usd",
    "population_total",
    "gdp_growth_annual_pct",
    "trade_pct_of_gdp"
]

df_clean = df[columns_to_keep].copy()

print("Created working cleaned World Bank dataframe.")
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns kept: {df_clean.shape[1]}")

Created working cleaned World Bank dataframe.
Rows: 2,170
Columns kept: 7


In [15]:
df_clean.head()

,reporter_iso3,reporter_name,reference_year,gdp_current_usd,population_total,gdp_growth_annual_pct,trade_pct_of_gdp
0,ABW,Aruba,2015,2.962907e+09,107906,3.611479,140.493001
1,ABW,Aruba,2016,2.983637e+09,108727,1.234335,136.887263
2,ABW,Aruba,2017,3.092428e+09,108735,3.493430,138.716620
3,ABW,Aruba,2018,3.276188e+09,108908,3.212471,138.824972
4,ABW,Aruba,2019,3.347562e+09,109203,1.229144,137.923249


#### Cleaning string columns

In [16]:
df_clean["reporter_iso3"] = (
    df_clean["reporter_iso3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_clean["reporter_name"] = (
    df_clean["reporter_name"]
    .astype("string")
    .str.strip()
)

#### Converting data types properly

In [17]:
df_clean["reference_year"] = pd.to_numeric(
    df_clean["reference_year"],
    errors="coerce"
).astype("Int64")

df_clean["population_total"] = pd.to_numeric(
    df_clean["population_total"],
    errors="coerce"
).astype("Int64")

numeric_float_cols = [
    "gdp_current_usd",
    "gdp_growth_annual_pct",
    "trade_pct_of_gdp"
]

for col in numeric_float_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2170 entries, 0 to 2169
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   reporter_iso3          2170 non-null   string 
 1   reporter_name          2170 non-null   string 
 2   reference_year         2170 non-null   Int64  
 3   gdp_current_usd        2088 non-null   float64
 4   population_total       2170 non-null   Int64  
 5   gdp_growth_annual_pct  2084 non-null   float64
 6   trade_pct_of_gdp       1820 non-null   float64
dtypes: Int64(2), float64(3), string(2)
memory usage: 123.0 KB


#### Validate country ISO codes

In [18]:
invalid_iso_rows = df_clean[
    df_clean["reporter_iso3"].isna() |
    (df_clean["reporter_iso3"].str.len() != 3)
]

print(f"Rows with invalid ISO3 codes: {len(invalid_iso_rows):,}")

# Checking if there are any rows with invalid ISO3 codes
invalid_iso_rows.head()

Rows with invalid ISO3 codes: 0


,reporter_iso3,reporter_name,reference_year,gdp_current_usd,population_total,gdp_growth_annual_pct,trade_pct_of_gdp


#### Validate year coverage

In [19]:
print("Minimum reference year:", df_clean["reference_year"].min())
print("Maximum reference year:", df_clean["reference_year"].max())

print("\nReference year value counts:")
df_clean["reference_year"].value_counts().sort_index()

Minimum reference year: 2015
Maximum reference year: 2024

Reference year value counts:


reference_year
2015    217
2016    217
2017    217
2018    217
2019    217
2020    217
2021    217
2022    217
2023    217
2024    217
Name: count, dtype: Int64

#### Reviewing missing values after cleaning

In [20]:
clean_missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_pct": (df_clean.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

clean_missing_summary

,missing_count,missing_pct
trade_pct_of_gdp,350,16.13
gdp_growth_annual_pct,86,3.96
gdp_current_usd,82,3.78
reference_year,0,0.00
reporter_name,0,0.00
reporter_iso3,0,0.00
population_total,0,0.00


In [21]:
# Checking for duplicate country-year combinations after cleaning
duplicate_country_year_rows = df_clean.duplicated(
    subset=["reporter_iso3", "reference_year"]
).sum()

print(f"Duplicate country-year rows after cleaning: {duplicate_country_year_rows:,}")

Duplicate country-year rows after cleaning: 0


In [25]:
# Creating new columns to indicate the presence of data in key columns

df_clean["has_gdp_data"] = df_clean["gdp_current_usd"].notna().astype("boolean")
df_clean["has_population_data"] = df_clean["population_total"].notna().astype("boolean")
df_clean["has_gdp_growth_data"] = df_clean["gdp_growth_annual_pct"].notna().astype("boolean")
df_clean["has_trade_pct_gdp_data"] = df_clean["trade_pct_of_gdp"].notna().astype("boolean")

df_clean.head()

,reporter_iso3,reporter_name,reference_year,gdp_current_usd,population_total,gdp_growth_annual_pct,trade_pct_of_gdp,has_gdp_data,has_population_data,has_gdp_growth_data,has_trade_pct_gdp_data
0,ABW,Aruba,2015,2.962907e+09,107906,3.611479,140.493001,True,True,True,True
1,ABW,Aruba,2016,2.983637e+09,108727,1.234335,136.887263,True,True,True,True
2,ABW,Aruba,2017,3.092428e+09,108735,3.493430,138.716620,True,True,True,True
3,ABW,Aruba,2018,3.276188e+09,108908,3.212471,138.824972,True,True,True,True
4,ABW,Aruba,2019,3.347562e+09,109203,1.229144,137.923249,True,True,True,True


In [26]:
# Finalizing the cleaned dataset by reordering columns and sorting by country and year
final_column_order = [
    "reporter_iso3",
    "reporter_name",
    "reference_year",
    "gdp_current_usd",
    "population_total",
    "gdp_growth_annual_pct",
    "trade_pct_of_gdp",
    "has_gdp_data",
    "has_population_data",
    "has_gdp_growth_data",
    "has_trade_pct_gdp_data"
]

df_final = df_clean[final_column_order].copy()

df_final = df_final.sort_values(
    ["reporter_iso3", "reference_year"]
).reset_index(drop=True)

df_final.head()

,reporter_iso3,reporter_name,reference_year,gdp_current_usd,population_total,gdp_growth_annual_pct,trade_pct_of_gdp,has_gdp_data,has_population_data,has_gdp_growth_data,has_trade_pct_gdp_data
0,ABW,Aruba,2015,2.962907e+09,107906,3.611479,140.493001,True,True,True,True
1,ABW,Aruba,2016,2.983637e+09,108727,1.234335,136.887263,True,True,True,True
2,ABW,Aruba,2017,3.092428e+09,108735,3.493430,138.716620,True,True,True,True
3,ABW,Aruba,2018,3.276188e+09,108908,3.212471,138.824972,True,True,True,True
4,ABW,Aruba,2019,3.347562e+09,109203,1.229144,137.923249,True,True,True,True


In [27]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2170 entries, 0 to 2169
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   reporter_iso3           2170 non-null   string 
 1   reporter_name           2170 non-null   string 
 2   reference_year          2170 non-null   Int64  
 3   gdp_current_usd         2088 non-null   float64
 4   population_total        2170 non-null   Int64  
 5   gdp_growth_annual_pct   2084 non-null   float64
 6   trade_pct_of_gdp        1820 non-null   float64
 7   has_gdp_data            2170 non-null   boolean
 8   has_population_data     2170 non-null   boolean
 9   has_gdp_growth_data     2170 non-null   boolean
 10  has_trade_pct_gdp_data  2170 non-null   boolean
dtypes: Int64(2), boolean(4), float64(3), string(2)
memory usage: 140.0 KB


#### Save the cleaned analytics-ready World Bank dataset

In [28]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_file_name = f"world_bank_country_year_clean_analytics_ready_{timestamp}.csv"
output_file_path = os.path.join(PROCESSED_OUTPUT_DIR, output_file_name)

df_final.to_csv(output_file_path, index=False)

print("Cleaned World Bank analytics-ready file saved successfully.")
print(output_file_path)

Cleaned World Bank analytics-ready file saved successfully.
C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\processed_data\world_bank_country_year_clean_analytics_ready_20260709_021503.csv


#### Final summary of the cleaned output

In [29]:
print("========== CLEANED WORLD BANK DATA SUMMARY ==========")
print(f"Rows: {df_final.shape[0]:,}")
print(f"Columns: {df_final.shape[1]:,}")
print(f"Country count: {df_final['reporter_iso3'].nunique():,}")
print(f"Reference year range: {df_final['reference_year'].min()} to {df_final['reference_year'].max()}")
print(f"Cleaned output file: {output_file_path}")

========== CLEANED WORLD BANK DATA SUMMARY ==========
Rows: 2,170
Columns: 11
Country count: 217
Reference year range: 2015 to 2024
Cleaned output file: C:\Users\hp\OneDrive\Desktop\global-trade-supply-chain-intelligence\data\processed_data\world_bank_country_year_clean_analytics_ready_20260709_021503.csv
